The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

1\.d Find out which is the letter with most counts.

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [1]:
# EXAM EXERCISE

#!/bin/bash

cd $HOME #change working directory to $HOME
rm students/* #delete all elements int students/ if already present elements
rm -d students/ #delete the directory students, -d is needed
mkdir -p students #makes dir if not existent
if [ ! -f "./students/LCP_22-23_students.csv" ] #if not already present, as in if NOT -find PATH
then
    wget -v --tries=1 https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv --directory-prefix="./students" #import file in ./students
fi # if is composed of if, then, fi
cd students # pass to students directory
touch LCP_22-23_PoD_students.csv #create if not existing
touch LCP_22-23_Physics_students.csv #create if not existing
grep "PoD" LCP_22-23_students.csv > LCP_22-23_PoD_students.csv #copy only lines including PoD from file A to file B
grep "Physics" LCP_22-23_students.csv > LCP_22-23_Physics_students.csv #copy only lines including Physics from file A to file B
max=0 #setting a local variable (lowercase) to search maximum
max_L='A' #setting a local variable (lowercase) to search maximum corresponding letter
for i in {A..Z} #i assumes values in the alphabet capital letters
do
    # to j is assgined the return of the previous computed part inside `, the left part is executed then passed as input to the right part
    j=`grep -v -e  "^Family" LCP_22-23_students.csv | grep -c "^$i" LCP_22-23_students.csv` # first gives lines without Family Name (-v exludes -e is followed by the relative element), second counts starting with A
    echo "$i : $j" #print out both variable; the $ is required in order to obtain the stored value, but works only inside double apix ""
    if [ $j -gt $max ] #compare values, not string (> is for strings in bash, -gt is grater than for numbers)
    then
        max=$j #updating maximum
        max_L=$i #updating best letter
    fi
done #sintax for for is for do done
echo "Max surname starting letter $max_L with $max entries" # print found output
lines=`grep '' -c LCP_22-23_students.csv` #counts all lines in file
i=2 #starting from the third line
while [ $i -le $lines ] #for each line, it will print it inside a different file, according to the modulo 18
do
    let g=($i-1)%18 #real computation for variables needs let or (( ))
    file="Group$g.csv" #naming scheme that uses value of i, there are ""
    touch $file #create file
    awk NR==$i LCP_22-23_students.csv >> $file #selected line with awk
    let i+=1
done
#spacing is always wrong in bash, unless is necessary

SyntaxError: invalid decimal literal (971401427.py, line 9)

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

2\.b How many even numbers are there?

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

In [ ]:
# EXAM EXERCISE

#!/bin/bash

grep -v '^#' data.csv | sed -e 's/,//g' > data.txt #take the lines in data.csv that don't start with #, substitute (sed) element (-e) from , to nothing
even=0 #starting to count even
for i in `cat data.txt` #for element in the dump of the file, so the single number can be selected
do
    #IN BASH TRUE IS 0 AND 1 IS FALSE, [ ] make it interpret as bool, but (( )) is for numbers,  then converted to bools that way
    if (( $i%2 )) #%2 gives 0 if even, that correspond to true in bash
    then
        let even+=1 #update counter, let is mandatory
    fi
done
echo "even numbers = $even"
#setting new local variable to count
m=0
l=0
 #getting a set value for sigma, $ pass the value instead of the command, |bc make it compute in floating point operation, '' are ok because there are no variables
sigma=$( echo 'scale=6;100*sqrt(3)/2.0' | bc)
FILE="data.txt" #global variable FILE
lines=`grep '' -c $FILE` #counts all lines in file
i=1 #starts from second line
while [ $i -le $lines ] #cycle over number of lines
do
    line=`awk NR==$i $FILE` #line is the string of the single i line in FILE
    IFS=' ' read -r X Y Z x y z <<< "$line" #set variables X, Y, Z, x, y, z from the line using separator ' ', read requires <<<
    d=$( echo "scale=6;sqrt($X*$X+$Y*$Y+$Z*$Z)" | bc) # assign to d the distance computed
    #echo $d
    if [ `echo "$d < $sigma" | bc` -eq 0 ] #this is a bool op. because of -eq, check if d<=sigma, using bc to do calculation
    then
        let m++ #update more than
    else
        let l++ #update less than
    fi
    let i++ #update i
done
echo "there are $m of distance grater than $sigma"
echo "there are $l of distance smaller than $sigma"
if [ -z $1 ] #1 is the first input when calling the script, -z check if it is a NULL, apparently -z is bool op
then
    echo "This program requires an input for normalization"
    exit
fi
if [ $1 -lt 1 ] #check the normalization if <= 1
then
    echo "This program requires an input grater than 1 for normalization"
    exit
fi
for (( i=1; i<=$1; i++ )) #computing operation (())
do
    # DIFFERENCE -v for awk is value, for grep is "without"
    #-v passes i as a variable to awk; cycle over NF Number of Fields; $j content of j field, 
    #check if field equal to number and end ($); then print as float j/i
    awk -v i="$i" '{for(j=1;j<=NF;j++) if($j~/^[0-9]+$/) $j=sprintf("%.1f",$j/i)}1' data.txt > "data$i.csv"
    done
#[0-9]* for zero or more numbers; [0-9][0-9]* one or more; etc. those are patterns with 2 and 3 element resp. \1 \2 \3
# % echo "123 abc" | sed 's/[0-9]*/& &/'
# 123 123 abc
# sed y is like tr